# MedGuide | تنزيل MedPackID v1 وفحص الصور

شغّلي **Runtime → Run all** واسمحي بالوصول إلى Google Drive. التنزيل يتم من خادم Colab إلى Drive داخل `MyDrive/MedGuideData/MedPackID_v1`. أرشيف الصور حوالي **3.03 GB**. يمكنك إعادة تشغيل الدفتر؛ الملفات المكتملة محفوظة، ويحاول استئناف الملف الجزئي إذا كان الخادم يدعم ذلك.

**المصدر:** [MedPackID v1](https://data.mendeley.com/datasets/d23kndp3sv/1) — Qory Hidayati, Hendra Kusuma, Muhammad Attamimi. DOI: 10.17632/d23kndp3sv.1. CC BY 4.0.

يعرض الدفتر أمثلة لصناديق YOLO من ملفات المصدر ويحفظ تقرير المطابقة. تسميات العلبة والحبة تصف فئة الجسم، وهوية الدواء لكل صورة تحتاج تحققًا مستقلًا قبل تقسيم البيانات حسب المنتج.

تم فحص 1,444 زوجًا من الصور والتسميات من المصدر في بيئة العمل. راجعي `reports/medpack_inspection.json`. هذه الجلسة لا تحتوي مخرجات تدريب.

In [ ]:
from google.colab import drive
from pathlib import Path
import requests

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/MedGuideData/MedPackID_v1')
root.mkdir(parents=True, exist_ok=True)

DATASET_ID = 'd23kndp3sv'
VERSION = 1
FILES = ('labels.zip', 'data.yaml', 'labels.txt', 'license.txt', 'images.zip', 'drug_candidates.csv', 'drug_name_dose_function.csv')

# رابط تنزيل مباشر اختياري لكل ملف إذا تغيّرت واجهة Mendeley.
# الاستعمال العادي يترك هذا القاموس فارغًا.
DIRECT_LINKS = {}


In [ ]:
import hashlib
import json
import subprocess
import zipfile
from urllib.parse import urlsplit

def get_file_metadata():
    url = f'https://data.mendeley.com/public-api/datasets/{DATASET_ID}'
    errors = []
    for endpoint, listing in ((url, False), (url + '/files', True)):
        try:
            params = {'version': VERSION}
            if listing:
                params['folder_id'] = 'root'
            reply = requests.get(endpoint, params=params, timeout=(15, 60))
            reply.raise_for_status()
            payload = reply.json()
            if isinstance(payload, dict):
                if payload.get('version') is not None and str(payload['version']) != str(VERSION):
                    raise ValueError('Wrong dataset version')
                records = payload.get('files', [])
            else:
                records = payload
            files = {}
            for record in records:
                name = record.get('filename')
                info = record.get('content_details') or {}
                link = info.get('download_url')
                if name in FILES and link:
                    files[name] = (link, info.get('size'), info.get('sha256_hash'))
            if all(name in files or name in DIRECT_LINKS for name in FILES):
                return files
            errors.append('Missing download links')
        except (requests.RequestException, ValueError, TypeError) as exc:
            errors.append(type(exc).__name__)
    raise RuntimeError('تعذّر جلب روابط Mendeley العامة. أعيدي تشغيل الخلية أو ضعي روابط الملفات المباشرة في DIRECT_LINKS. ' + ', '.join(errors))

def verify_file(path, name, size, expected_hash):
    if not path.is_file() or path.stat().st_size == 0:
        return False
    if size is not None and path.stat().st_size != int(size):
        return False
    if expected_hash:
        with path.open('rb') as file:
            if hashlib.file_digest(file, 'sha256').hexdigest().lower() != expected_hash.lower():
                return False
    if name.endswith('.zip'):
        return zipfile.is_zipfile(path)
    return b'<html' not in path.read_bytes()[:512].lower() and b'<!doctype' not in path.read_bytes()[:512].lower()

def download_asset(name, url, size, checksum):
    if urlsplit(url).scheme != 'https':
        raise ValueError(f'Expected HTTPS download link for {name}')
    target = root / name
    if verify_file(target, name, size, checksum):
        print(f'موجود في Drive: {name}')
        return
    partial = root / (name + '.part')
    if not verify_file(partial, name, size, checksum) or size is None and not checksum:
        args = ['curl', '--fail', '--location', '--retry', '3', '--connect-timeout', '25',
                '--proto', '=https', '--proto-redir', '=https', '--progress-bar']
        print(f'تحميل {name}', flush=True)
        command = args + ['--output', str(partial), url]
        resume = partial.exists() and partial.stat().st_size > 0
        result = subprocess.run(command + (['--continue-at', '-'] if resume else []), check=False)
        if resume and result.returncode == 33:
            result = subprocess.run(command, check=False)
        result.check_returncode()
    if not verify_file(partial, name, size, checksum):
        partial.unlink(missing_ok=True)
        raise RuntimeError(f'فشل التحقق من الملف: {name}. أعيدي تشغيل الخلية.')
    partial.replace(target)
    print(f'تم الحفظ: {target.name} ({target.stat().st_size:,} bytes)')

metadata = {} if all(name in DIRECT_LINKS for name in FILES) else get_file_metadata()
for name in FILES:
    url, size, checksum = metadata.get(name, (None, None, None))
    if name in DIRECT_LINKS:
        url = DIRECT_LINKS[name]
    download_asset(name, url, size, checksum)


In [ ]:
from collections import Counter
from io import BytesIO
from pathlib import PurePosixPath
from PIL import Image, ImageDraw
from IPython.display import display

def index_members(archive, suffixes):
    result = {}
    for name in archive.namelist():
        path = PurePosixPath(name)
        if path.suffix.lower() in suffixes:
            if path.stem in result:
                raise ValueError(f'Duplicate filename stem: {path.stem}')
            result[path.stem] = name
    return result

def inspect_archives(images_path, labels_path, preview_count=6):
    with zipfile.ZipFile(images_path) as images, zipfile.ZipFile(labels_path) as labels:
        pictures = index_members(images, {'.jpg', '.jpeg', '.png', '.webp'})
        annotations = index_members(labels, {'.txt'})
        paired = sorted(pictures.keys() & annotations.keys())
        report = {
            'dataset': 'MedPackID v1', 'doi': '10.17632/d23kndp3sv.1',
            'images': len(pictures), 'labels': len(annotations), 'paired': len(paired),
            'missing_images': sorted(annotations.keys() - pictures.keys()),
            'missing_labels': sorted(pictures.keys() - annotations.keys()),
        }
        counts = Counter()
        previews = 0
        for key in paired:
            boxes = []
            for line in labels.read(annotations[key]).decode('utf-8-sig').splitlines():
                if not line.strip():
                    continue
                category, *values = line.split()
                x, y, w, h = map(float, values)
                edges = (x-w/2, y-h/2, x+w/2, y+h/2)
                if category not in {'0', '1'} or not (w > 0 and h > 0 and min(edges) >= -1e-5 and max(edges) <= 1.00001):
                    raise ValueError(f'Invalid bounding box: {annotations[key]}')
                counts[category] += 1
                boxes.append((category, edges))
            if previews >= preview_count or not any(category == '0' for category, _ in boxes):
                continue
            with Image.open(BytesIO(images.read(pictures[key]))) as original:
                picture = original.convert('RGB')
            picture.thumbnail((520, 520))
            draw = ImageDraw.Draw(picture)
            for category, (left, top, right, bottom) in boxes:
                draw.rectangle((left*picture.width, top*picture.height, right*picture.width, bottom*picture.height),
                               outline='lime' if category == '0' else 'orange', width=3)
            print(pictures[key])
            display(picture)
            previews += 1
        report['box_counts'] = {'medicine_packaging': counts['0'], 'pill': counts['1']}
        return report

report = inspect_archives(root / 'images.zip', root / 'labels.zip')
report_path = root / 'inspection_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print({key: value for key, value in report.items() if not key.startswith('missing_')})
print('التقرير:', report_path)
if not report['paired'] or report['missing_images'] or report['missing_labels']:
    raise ValueError('توجد ملفات غير متطابقة. راجعي أسماءها في التقرير.')


## تدريب YOLO11n لكشف العلب

اختاري GPU في Colab ثم شغّلي الخلايا. الكود يستخدم فئة واحدة للعلبة. صور الحبوب فقط تُستخدم كأمثلة سلبية؛ التسميات الفارغة من المصدر تُستبعد احتياطًا.

للتدريب المعتمد على تقسيم المنتجات، ضعي `identities.csv` في مجلد البيانات بأعمدة `file_name,product_id` لكل صورة مستخدمة. عند غيابه، يمكن إجراء تدريب استكشافي بتقسيم الناشر بعد تفعيل `ALLOW_PILOT`. مقاييسه لا تثبت التعميم على منتجات جديدة، ولا يُصدَّر منه نموذج للهاتف. خط الأساس يقيس اقتراحات الأجسام العامة من أوزان COCO بعد جمع الفئات، لأن COCO لا يوفر فئة علبة دواء مخصصة.

يجب أن يكون لكل صورة في المجموعات المعتمدة هوية مراجَعة ومصدر في `identities.csv`، بما فيها صور الحبوب إن استُخدمت. للصور التي تشتمل على منتجات مختلفة أضيفي المعرفات مفصولة بـ `|`، وستبقى كل المنتجات المتداخلة في نفس المجموعة. ملفات الأسماء من الناشر لا تربط أسماء المنتجات بكل صورة.

In [ ]:
import sys
import subprocess

ALLOW_PILOT = False
EPOCHS = 40
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9'])


In [ ]:
import csv
import random
import shutil
import hashlib
import tempfile
import warnings
from collections import Counter
import yaml
import torch
import ultralytics

splits = ('train', 'val', 'test')
identity_file = root / 'identities.csv'
verified_split = identity_file.is_file()
if not verified_split and not ALLOW_PILOT:
    raise RuntimeError('أضيفي identities.csv للتقسيم حسب المنتج أو فعّلي ALLOW_PILOT لتجربة استكشافية.')
identities = {}
if verified_split:
    with identity_file.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        if not {'file_name', 'product_id', 'identity_source'} <= set(reader.fieldnames or []):
            raise ValueError('identities.csv requires file_name,product_id,identity_source')
        for row in reader:
            key = PurePosixPath(row['file_name']).stem
            product = tuple(sorted({item.strip() for item in row['product_id'].split('|') if item.strip()}))
            if not product or not row['identity_source'].strip() or key in identities:
                raise ValueError(f'Invalid or duplicate identity: {key}')
            identities[key] = product

with zipfile.ZipFile(root / 'images.zip') as images, zipfile.ZipFile(root / 'labels.zip') as labels:
    image_index = index_members(images, {'.jpg', '.jpeg', '.png', '.webp'})
    label_index = index_members(labels, {'.txt'})
    if image_index.keys() != label_index.keys():
        raise ValueError('راجع تقرير الفحص قبل التدريب.')
    records = []
    for key in sorted(label_index):
        member = label_index[key]
        lines = labels.read(member).decode('utf-8-sig').splitlines()
        lines = [line.strip() for line in lines if line.strip()]
        if not lines:
            continue
        split = PurePosixPath(member).parent.name
        if split not in splits:
            raise ValueError(f'Unknown publisher split: {member}')
        if verified_split and key not in identities:
            raise ValueError(f'Missing verified product identity: {key}')
        records.append((key, split, [line for line in lines if line.split()[0] == '0']))
    assignment = {}
    if verified_split:
        components = {pid: {pid} for key, _, _ in records for pid in identities[key]}
        for key, _, _ in records:
            joined = set().union(*(components[pid] for pid in identities[key]))
            for pid in joined:
                components[pid] = joined
        group_by_product = {pid: min(group) for pid, group in components.items()}
        groups = sorted(set(group_by_product.values()))
        if len(groups) < 3:
            raise ValueError('Need at least three verified products.')
        random.Random(42).shuffle(groups)
        holdout = max(1, round(.15 * len(groups)))
        assignment = {group: 'test' if i < holdout else 'val' if i < holdout * 2 else 'train'
                      for i, group in enumerate(groups)}
    dataset = Path(tempfile.mkdtemp(prefix='medguide-yolo-', dir='/content'))
    for split in splits:
        for folder in ('images', 'labels'):
            (dataset / folder / split).mkdir(parents=True)
    hashes = {}
    manifest = []
    counts = Counter()
    for key, publisher_split, boxes in records:
        split = assignment[group_by_product[identities[key][0]]] if verified_split else publisher_split
        data = images.read(image_index[key])
        digest = hashlib.sha256(data).hexdigest()
        if digest in hashes:
            if hashes[digest] != (split, identities.get(key), tuple(boxes)):
                raise ValueError(f'Identical image with conflicting split, identity or labels: {key}')
            continue
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('error', Image.DecompressionBombWarning)
                with Image.open(BytesIO(data)) as image:
                    if image.getexif().get(274, 1) != 1:
                        continue
                    image.load()
        except (OSError, Image.DecompressionBombWarning, Image.DecompressionBombError):
            continue
        hashes[digest] = (split, identities.get(key), tuple(boxes))
        suffix = PurePosixPath(image_index[key]).suffix.lower()
        (dataset / 'images' / split / (key + suffix)).write_bytes(data)
        (dataset / 'labels' / split / (key + '.txt')).write_text('\n'.join(boxes), encoding='utf-8')
        counts[split] += bool(boxes)
        manifest.append([key, split, identities.get(key), digest, boxes])

if not all(counts[s] for s in splits):
    raise ValueError(f'Each split needs labeled medicine boxes: {counts}')
tag = hashlib.sha256(json.dumps(manifest).encode()).hexdigest()[:12]
mode = 'product_split' if verified_split else 'pilot_publisher_split'
training_root = root / 'training' / f'{mode}_{tag}_e{EPOCHS}'
training_root.mkdir(parents=True, exist_ok=True)
config = dataset / 'dataset.yaml'
config.write_text(yaml.safe_dump({'path': str(dataset), **{s: f'images/{s}' for s in splits},
                                  'names': {0: 'medicine_box'}}))
(training_root / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2))
(training_root / 'versions.json').write_text(json.dumps({'torch':torch.__version__, 'ultralytics':ultralytics.__version__,
    'python':sys.version, 'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}))
print(mode, dict(counts), training_root)


In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    raise RuntimeError('فعّلي GPU في Colab لتشغيل التدريب.')
args = dict(data=str(config), imgsz=640, device=device, batch=8, workers=2, project=str(training_root))
baseline_path = training_root / 'baseline_val.json'
if not baseline_path.exists():
    baseline = YOLO('yolo11n.pt').val(**args, name='baseline', split='val', single_cls=True, exist_ok=True)
    baseline_path.write_text(json.dumps(dict(baseline.results_dict) | {'speed_ms':baseline.speed}, indent=2))

checkpoints = training_root / 'fit' / 'weights'
best = checkpoints / 'best.pt'
last = checkpoints / 'last.pt'
done = training_root / 'fit_complete'
if not done.exists():
    if last.is_file():
        YOLO(str(last)).train(resume=True, data=str(config), device=device)
    else:
        YOLO('yolo11n.pt').train(**args, name='fit', exist_ok=True, epochs=EPOCHS,
            seed=42, deterministic=True, patience=12, degrees=12, perspective=0.0005,
            hsv_v=0.35, hsv_s=0.3, scale=0.45, mixup=0, close_mosaic=8)
    if not best.is_file():
        raise RuntimeError('Training ended without best.pt')
    done.touch()

split = 'test' if verified_split else 'val'
metrics = YOLO(str(best)).val(**args, name=f'final_{split}', split=split, exist_ok=True)
path = training_root / f'metrics_{split}.json'
path.write_text(json.dumps(dict(metrics.results_dict) | {'speed_ms':metrics.speed, 'mode':mode}, indent=2))
print(best, path)


In [ ]:
# تصدير بعد التدريب المقسّم حسب المنتج. المعايرة من train والتقييم من test.
if verified_split:
    mobile = training_root / 'mobile'
    mobile.mkdir(exist_ok=True)
    report_path = mobile / 'yolo_export_report.json'
    report_path.unlink(missing_ok=True)
    export_report = {'mode': mode, 'native_map50': float(metrics.box.map50)}
    for fmt, filename, extra in [
        ('onnx', 'yolo11n_medguide.onnx', {'opset': 17, 'dynamic': False, 'nms': False}),
        ('tflite', 'yolo11n_medguide_int8.tflite', {'int8': True, 'data': str(config), 'split': 'train'}),
    ]:
        exported = Path(YOLO(str(best)).export(format=fmt, imgsz=640, device='cpu', **extra))
        measured = YOLO(str(exported), task='detect').val(
            data=str(config), split='test', imgsz=640, device='cpu', batch=1,
            project=str(training_root), name=f'{fmt}_test', exist_ok=True)
        drop = float(metrics.box.map50 - measured.box.map50)
        report = dict(measured.results_dict) | {'speed_ms': measured.speed, 'map50_drop': drop,
                                                'passed_two_point_gate': drop <= .02}
        (training_root / f'metrics_{fmt}_test.json').write_text(json.dumps(report, indent=2))
        if drop > .02:
            print(f'{fmt}: تجاوز انخفاض mAP50 نقطتين. راجعي المعايرة قبل الدمج.')
            (mobile / filename).unlink(missing_ok=True)
            continue
        destination = mobile / filename
        shutil.copy2(exported, destination)
        with destination.open('rb') as stream:
            export_report[f'{fmt}_sha256'] = hashlib.file_digest(stream, 'sha256').hexdigest()
        export_report[f'{fmt}_map50'] = float(measured.box.map50)
        if 'onnx_sha256' in export_report:
            report_path.write_text(json.dumps(export_report, indent=2))
else:
    print('التصدير مؤجل إلى توفر هويات المنتجات والتحقق من التقسيم.')